In [1]:
# ============================================================
# SetStop-KGQA Notebook 06
# Fully End-to-End Test Evaluation
# Complete single-cell Google Colab code
# ============================================================

!pip -q install -U \
    transformers \
    peft \
    accelerate \
    bitsandbytes \
    scikit-learn \
    safetensors

# ============================================================
# 1. IMPORTS
# ============================================================

import collections
import csv
import gc
import hashlib
import json
import random
import re
import time

from pathlib import Path

import numpy as np
import torch
import transformers

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
)

from peft import PeftModel
from google.colab import drive


# ============================================================
# 2. CLEAN RUNTIME AND MOUNT DRIVE
# ============================================================

for variable_name in [
    "controller",
    "controller_base",
    "answer_model",
]:
    if variable_name in globals():
        del globals()[variable_name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

drive.mount("/content/drive")


# ============================================================
# 3. CONFIGURATION
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ROOT = Path(
    "/content/drive/MyDrive/SetStop_KGQA"
)

DATA_DIR = (
    ROOT / "data/extracted/webqsp"
)

CONTROLLER_MODEL_PATH = (
    ROOT
    / "models/setstop_qwen_gain_controller_v1"
)

ANSWER_MODEL_PATH = (
    ROOT
    / "models/answer_node_minilm_v1"
)

CONTROLLER_THRESHOLD_PATH = (
    ROOT
    / "outputs/controller_training_v1/"
      "dev_stop_threshold_v1.json"
)

ANSWER_THRESHOLD_PATH = (
    ROOT
    / "outputs/answer_recognizer_v1/"
      "answer_recognizer_threshold_v1.json"
)

ORACLE_EVALUATION_PATH = (
    ROOT
    / "outputs/test_evaluation_v1/"
      "complete_test_evaluation_v1.json"
)

OUTPUT_ROOT = (
    ROOT
    / "outputs/end_to_end_test_v1"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

CONTROLLER_BASE_MODEL = (
    "Qwen/Qwen2.5-0.5B-Instruct"
)

MAX_HOPS = 3
MAX_ANSWERS = 50
MAX_ACTIONS = 12
MAX_FRONTIER_CANDIDATES = 16

CONTROLLER_BATCH_SIZE = 64
ANSWER_BATCH_SIZE = 128

GAIN_VALUES = np.asarray(
    [0.0, 1.0, 2.0, 3.5],
    dtype=np.float64
)

required_paths = [
    DATA_DIR / "test_simple.json",
    DATA_DIR / "entities.txt",
    DATA_DIR / "relations.txt",
    CONTROLLER_MODEL_PATH,
    ANSWER_MODEL_PATH,
    CONTROLLER_THRESHOLD_PATH,
    ANSWER_THRESHOLD_PATH,
]

for path in required_paths:
    assert path.exists(), (
        f"Missing required input:\n{path}"
    )

controller_threshold_data = json.loads(
    CONTROLLER_THRESHOLD_PATH.read_text(
        encoding="utf-8"
    )
)

answer_threshold_data = json.loads(
    ANSWER_THRESHOLD_PATH.read_text(
        encoding="utf-8"
    )
)

CONTROLLER_THRESHOLD = float(
    controller_threshold_data[
        "selected_threshold"
    ]
)

ANSWER_THRESHOLD = float(
    answer_threshold_data[
        "selected_threshold"
    ]
)

assert abs(
    CONTROLLER_THRESHOLD - 0.71
) < 1e-6

assert abs(
    ANSWER_THRESHOLD - 0.67
) < 1e-6

print("=" * 90)
print("SETSTOP-KGQA FULL END-TO-END EVALUATION")
print("=" * 90)
print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("Controller threshold:", CONTROLLER_THRESHOLD)
print("Answer threshold:", ANSWER_THRESHOLD)
print("Maximum actions:", MAX_ACTIONS)
print("Frontier candidates:", MAX_FRONTIER_CANDIDATES)

assert torch.cuda.is_available(), (
    "Select a T4 GPU runtime."
)

GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME)
print("=" * 90)


# ============================================================
# 4. DATA HELPERS
# ============================================================

def read_vocabulary(path):
    with open(path, encoding="utf-8") as file:
        return [
            line.rstrip("\n")
            for line in file
        ]


ENTITY_NAMES = read_vocabulary(
    DATA_DIR / "entities.txt"
)

RELATION_NAMES = read_vocabulary(
    DATA_DIR / "relations.txt"
)

ENTITY_TO_ID = {
    name: index
    for index, name
    in enumerate(ENTITY_NAMES)
}

RELATION_TO_ID = {
    name: index
    for index, name
    in enumerate(RELATION_NAMES)
}


def iter_jsonl(path):
    with open(path, encoding="utf-8") as file:
        for line_number, line in enumerate(
            file,
            start=1
        ):
            if not line.strip():
                continue

            try:
                yield json.loads(line)
            except Exception as error:
                raise ValueError(
                    f"Invalid JSON at "
                    f"{path}:{line_number}"
                ) from error


def entity_index(value):
    if isinstance(value, (int, np.integer)):
        return int(value)

    if isinstance(value, str):
        return ENTITY_TO_ID.get(value)

    if isinstance(value, dict):
        for key in ["kb_id", "id", "entity"]:
            if key in value:
                return entity_index(value[key])

    return None


def relation_index(value):
    if isinstance(value, (int, np.integer)):
        return int(value)

    if isinstance(value, str):
        return RELATION_TO_ID.get(value)

    return None


def get_question(example):
    for key in ["question", "text", "query"]:
        value = example.get(key)

        if isinstance(value, str):
            return value.strip()

    raise KeyError("Question field missing")


def get_topic_ids(example):
    raw = example.get(
        "entities",
        example.get("topic_entities", [])
    )

    values = {
        entity_index(item)
        for item in raw
    }

    return {
        item
        for item in values
        if item is not None
    }


def get_gold_ids(example):
    values = {
        entity_index(item)
        for item in example.get(
            "answers",
            []
        )
    }

    return {
        item
        for item in values
        if item is not None
    }


def get_graph(example):
    subgraph = example.get("subgraph", {})

    nodes = {
        entity_index(item)
        for item in subgraph.get(
            "entities",
            []
        )
    }

    nodes.discard(None)

    raw_edges = subgraph.get(
        "tuples",
        subgraph.get("triples", [])
    )

    edges = []

    for row in raw_edges:
        if isinstance(row, dict):
            subject = entity_index(
                row.get(
                    "subject",
                    row.get("s")
                )
            )

            relation = relation_index(
                row.get(
                    "relation",
                    row.get("r")
                )
            )

            object_entity = entity_index(
                row.get(
                    "object",
                    row.get("o")
                )
            )
        else:
            if len(row) != 3:
                continue

            subject = entity_index(row[0])
            relation = relation_index(row[1])
            object_entity = entity_index(row[2])

        if (
            subject is None
            or relation is None
            or object_entity is None
        ):
            continue

        subject = int(subject)
        relation = int(relation)
        object_entity = int(object_entity)

        edges.append(
            (subject, relation, object_entity)
        )

        nodes.add(subject)
        nodes.add(object_entity)

    return nodes, edges


def get_question_id(
    split,
    example,
    line_number
):
    for key in ["id", "qid", "question_id"]:
        if key in example:
            return str(example[key])

    digest = hashlib.sha1(
        get_question(example).encode(
            "utf-8"
        )
    ).hexdigest()[:12]

    return f"{split}_{line_number}_{digest}"


def relation_name(relation_id):
    if 0 <= relation_id < len(
        RELATION_NAMES
    ):
        return RELATION_NAMES[relation_id]

    return f"relation_{relation_id}"


def stable_seed(text):
    digest = hashlib.sha1(
        text.encode("utf-8")
    ).hexdigest()[:8]

    return SEED + int(digest, 16)


# ============================================================
# 5. GRAPH SEARCH
# ============================================================

def build_grouped_adjacency(edges):
    temporary = collections.defaultdict(
        lambda: collections.defaultdict(set)
    )

    for subject, relation, object_entity in edges:
        temporary[subject][relation].add(
            object_entity
        )

    return {
        source: {
            relation: sorted(targets)
            for relation, targets
            in relation_map.items()
        }
        for source, relation_map
        in temporary.items()
    }


def get_frontier(
    visited,
    node_depth,
    expanded,
    adjacency
):
    branches = []

    for source in sorted(visited):
        source_depth = node_depth.get(
            source,
            MAX_HOPS + 1
        )

        if source_depth >= MAX_HOPS:
            continue

        for relation, targets in adjacency.get(
            source,
            {}
        ).items():
            if (source, relation) in expanded:
                continue

            branches.append({
                "source": source,
                "relation": relation,
                "targets": targets,
                "source_depth": source_depth,
                "target_depth": source_depth + 1,
                "target_count": len(targets),
            })

    return branches


def tokenize_relation(text):
    return set(
        re.findall(
            r"[a-z0-9]+",
            text.lower().replace("_", " ")
        )
    )


def relation_overlap(question, relation_id):
    return len(
        tokenize_relation(question)
        & tokenize_relation(
            relation_name(relation_id)
        )
    )


def retrieve_frontier(
    branches,
    question
):
    ranked = sorted(
        branches,
        key=lambda branch: (
            -relation_overlap(
                question,
                branch["relation"]
            ),
            branch["source_depth"],
            -branch["target_count"],
            branch["relation"],
            branch["source"],
        )
    )

    return ranked[
        :MAX_FRONTIER_CANDIDATES
    ]


def expand_branch(
    branch,
    visited,
    node_depth,
    node_paths,
    expanded
):
    source = branch["source"]
    relation = branch["relation"]

    expanded.add(
        (source, relation)
    )

    source_path = node_paths.get(
        source,
        []
    )

    target_path = (
        source_path + [relation]
    )

    for target in branch["targets"]:
        visited.add(target)

        old_depth = node_depth.get(target)

        if (
            old_depth is None
            or branch["target_depth"] < old_depth
        ):
            node_depth[target] = (
                branch["target_depth"]
            )

            node_paths[target] = list(
                target_path
            )


def reachable_gold_from_branch(
    branch,
    adjacency,
    expanded,
    unseen_gold
):
    queue = collections.deque()
    best_depth = {}

    for target in branch["targets"]:
        best_depth[target] = (
            branch["target_depth"]
        )

        queue.append(
            (
                target,
                branch["target_depth"],
            )
        )

    reachable = set()

    while queue:
        node, depth = queue.popleft()

        if node in unseen_gold:
            reachable.add(node)

        if depth >= MAX_HOPS:
            continue

        for relation, targets in adjacency.get(
            node,
            {}
        ).items():
            if (node, relation) in expanded:
                continue

            next_depth = depth + 1

            for target in targets:
                previous_depth = best_depth.get(
                    target
                )

                if (
                    previous_depth is None
                    or next_depth < previous_depth
                ):
                    best_depth[target] = next_depth

                    queue.append(
                        (
                            target,
                            next_depth,
                        )
                    )

    return reachable


def remaining_gold_is_reachable(
    branches,
    adjacency,
    expanded,
    gold,
    truly_visited_answers
):
    unseen_gold = (
        gold - truly_visited_answers
    )

    if not unseen_gold:
        return False

    for branch in branches:
        if reachable_gold_from_branch(
            branch,
            adjacency,
            expanded,
            unseen_gold,
        ):
            return True

    return False


# ============================================================
# 6. LOAD BOTH MODELS
# ============================================================

controller_tokenizer = (
    AutoTokenizer.from_pretrained(
        CONTROLLER_MODEL_PATH,
        use_fast=True,
    )
)

if controller_tokenizer.pad_token is None:
    controller_tokenizer.pad_token = (
        controller_tokenizer.eos_token
    )

controller_quantization = (
    BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=
            torch.float16,
    )
)

controller_base = (
    AutoModelForSequenceClassification
    .from_pretrained(
        CONTROLLER_BASE_MODEL,
        num_labels=4,
        id2label={
            0: "0",
            1: "1",
            2: "2",
            3: "3_plus",
        },
        label2id={
            "0": 0,
            "1": 1,
            "2": 2,
            "3_plus": 3,
        },
        quantization_config=
            controller_quantization,
        device_map={"": 0},
        dtype=torch.float16,
    )
)

controller_base.config.pad_token_id = (
    controller_tokenizer.pad_token_id
)

controller = PeftModel.from_pretrained(
    controller_base,
    CONTROLLER_MODEL_PATH,
    is_trainable=False,
)

controller.eval()

answer_tokenizer = (
    AutoTokenizer.from_pretrained(
        ANSWER_MODEL_PATH,
        use_fast=True,
    )
)

answer_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        ANSWER_MODEL_PATH,
        dtype=torch.float16,
    )
)

answer_model = answer_model.to("cuda")
answer_model.eval()

print("\nModels loaded successfully.")


# ============================================================
# 7. INFERENCE HELPERS
# ============================================================

@torch.inference_mode()
def controller_probabilities(texts):
    outputs = []

    for start in range(
        0,
        len(texts),
        CONTROLLER_BATCH_SIZE
    ):
        batch = texts[
            start:
            start + CONTROLLER_BATCH_SIZE
        ]

        encoded = controller_tokenizer(
            batch,
            truncation=True,
            max_length=256,
            padding=True,
            return_tensors="pt",
        )

        encoded = {
            key: value.to(controller.device)
            for key, value
            in encoded.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            logits = controller(
                **encoded
            ).logits

        outputs.append(
            torch.softmax(
                logits.float(),
                dim=-1
            ).cpu().numpy()
        )

    return np.concatenate(
        outputs,
        axis=0
    )


@torch.inference_mode()
def answer_probabilities(texts):
    outputs = []

    for start in range(
        0,
        len(texts),
        ANSWER_BATCH_SIZE
    ):
        batch = texts[
            start:
            start + ANSWER_BATCH_SIZE
        ]

        encoded = answer_tokenizer(
            batch,
            truncation=True,
            max_length=192,
            padding=True,
            return_tensors="pt",
        )

        encoded = {
            key: value.to("cuda")
            for key, value
            in encoded.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            logits = answer_model(
                **encoded
            ).logits

        outputs.append(
            torch.softmax(
                logits.float(),
                dim=-1
            )[:, 1].cpu().numpy()
        )

    return np.concatenate(
        outputs,
        axis=0
    )


# ============================================================
# 8. MODEL INPUTS
# ============================================================

def make_controller_text(
    question,
    predicted_answer_count,
    visited_count,
    step,
    branch,
    history
):
    history_text = (
        " -> ".join(history[-6:])
        if history
        else "none"
    )

    return (
        f"Question: {question}\n"
        f"Answers found so far: "
        f"{predicted_answer_count}\n"
        f"Visited entities: {visited_count}\n"
        f"Search step: {step}\n"
        f"Source depth: "
        f"{branch['source_depth']}\n"
        f"Candidate relation: "
        f"{relation_name(branch['relation'])}\n"
        f"Candidate target count: "
        f"{branch['target_count']}\n"
        f"Recent relations: {history_text}"
    )


def make_answer_text(
    question,
    branch,
    source_path,
    target,
    adjacency
):
    relation_path = (
        source_path
        + [branch["relation"]]
    )

    path_text = " -> ".join(
        relation_name(relation)
        for relation in relation_path
    )

    return (
        f"Question: {question}\n"
        f"Candidate relation path: {path_text}\n"
        f"Candidate depth: "
        f"{branch['target_depth']}\n"
        f"Incoming relation: "
        f"{relation_name(branch['relation'])}\n"
        f"Entities returned by final relation: "
        f"{branch['target_count']}\n"
        f"Candidate outgoing relation count: "
        f"{len(adjacency.get(target, {}))}"
    )


def recognize_branch_answers(
    question,
    branch,
    node_paths,
    adjacency,
    node_answer_scores,
    predicted_answers
):
    source_path = node_paths.get(
        branch["source"],
        []
    )

    texts = [
        make_answer_text(
            question=question,
            branch=branch,
            source_path=source_path,
            target=target,
            adjacency=adjacency,
        )
        for target in branch["targets"]
    ]

    probabilities = answer_probabilities(
        texts
    )

    newly_predicted = set()

    for target, probability in zip(
        branch["targets"],
        probabilities
    ):
        previous_score = node_answer_scores.get(
            target,
            0.0
        )

        best_score = max(
            previous_score,
            float(probability)
        )

        node_answer_scores[target] = (
            best_score
        )

        if best_score >= ANSWER_THRESHOLD:
            if target not in predicted_answers:
                newly_predicted.add(target)

            predicted_answers.add(target)

    return newly_predicted


# ============================================================
# 9. RESULT CREATION
# ============================================================

def create_result(
    qid,
    policy,
    gold_answers,
    predicted_answers,
    truly_visited_answers,
    actions,
    retrieved_edges,
    over_search_actions,
    stop_reason,
    premature_stop,
    frontier_truncations
):
    true_positives = len(
        predicted_answers & gold_answers
    )

    false_positives = len(
        predicted_answers - gold_answers
    )

    false_negatives = len(
        gold_answers - predicted_answers
    )

    precision = (
        true_positives
        / len(predicted_answers)
        if predicted_answers
        else 0.0
    )

    recall = (
        true_positives
        / len(gold_answers)
    )

    if precision + recall == 0:
        answer_f1 = 0.0
    else:
        answer_f1 = (
            2 * precision * recall
            / (precision + recall)
        )

    return {
        "qid": qid,
        "policy": policy,
        "gold_answer_count":
            len(gold_answers),
        "predicted_answer_count":
            len(predicted_answers),
        "true_positive_count":
            true_positives,
        "false_positive_count":
            false_positives,
        "false_negative_count":
            false_negatives,
        "precision": float(precision),
        "recall": float(recall),
        "answer_f1": float(answer_f1),
        "exact_match": int(
            predicted_answers == gold_answers
        ),
        "complete_recall": int(
            recall == 1.0
        ),
        "incomplete_set": int(
            recall < 1.0
        ),
        "actions": int(actions),
        "retrieved_edges":
            int(retrieved_edges),
        "over_search_actions":
            int(over_search_actions),
        "premature_stop":
            int(premature_stop),
        "frontier_truncations":
            int(frontier_truncations),
        "stop_reason": stop_reason,
    }


# ============================================================
# 10. LEARNED TRAJECTORY
# ============================================================

def run_learned_trajectory(
    qid,
    example
):
    question = get_question(example)
    gold = get_gold_ids(example)
    topics = get_topic_ids(example)

    _, edges = get_graph(example)

    adjacency = build_grouped_adjacency(
        edges
    )

    visited = set(topics)

    node_depth = {
        topic: 0
        for topic in topics
    }

    node_paths = {
        topic: []
        for topic in topics
    }

    expanded = set()
    history = []

    node_answer_scores = {}
    predicted_answers = set()

    truly_visited_answers = (
        visited & gold
    )

    actions = 0
    retrieved_edges = 0
    over_search_actions = 0
    frontier_truncations = 0

    snapshots = []

    while actions < MAX_ACTIONS:
        all_branches = get_frontier(
            visited,
            node_depth,
            expanded,
            adjacency,
        )

        if not all_branches:
            break

        if (
            len(all_branches)
            > MAX_FRONTIER_CANDIDATES
        ):
            frontier_truncations += 1

        branches = retrieve_frontier(
            all_branches,
            question
        )

        controller_texts = [
            make_controller_text(
                question=question,
                predicted_answer_count=
                    len(predicted_answers),
                visited_count=len(visited),
                step=actions,
                branch=branch,
                history=history,
            )
            for branch in branches
        ]

        probabilities = (
            controller_probabilities(
                controller_texts
            )
        )

        positive_gain = (
            1.0 - probabilities[:, 0]
        )

        expected_gain = (
            probabilities @ GAIN_VALUES
        )

        maximum_positive_gain = float(
            np.max(positive_gain)
        )

        should_stop = (
            maximum_positive_gain
            < CONTROLLER_THRESHOLD
        )

        premature_if_stopped = False

        if should_stop:
            premature_if_stopped = (
                remaining_gold_is_reachable(
                    branches=all_branches,
                    adjacency=adjacency,
                    expanded=expanded,
                    gold=gold,
                    truly_visited_answers=
                        truly_visited_answers,
                )
            )

        snapshots.append({
            "actions": actions,
            "retrieved_edges":
                retrieved_edges,
            "predicted_answers":
                set(predicted_answers),
            "truly_visited_answers":
                set(truly_visited_answers),
            "over_search_actions":
                over_search_actions,
            "frontier_truncations":
                frontier_truncations,
            "maximum_positive_gain":
                maximum_positive_gain,
            "should_stop": should_stop,
            "premature_if_stopped":
                premature_if_stopped,
        })

        best_index = int(
            np.argmax(expected_gain)
        )

        chosen = branches[best_index]

        true_complete_before = (
            truly_visited_answers == gold
        )

        expand_branch(
            chosen,
            visited,
            node_depth,
            node_paths,
            expanded,
        )

        recognize_branch_answers(
            question=question,
            branch=chosen,
            node_paths=node_paths,
            adjacency=adjacency,
            node_answer_scores=
                node_answer_scores,
            predicted_answers=
                predicted_answers,
        )

        actions += 1
        retrieved_edges += (
            chosen["target_count"]
        )

        history.append(
            relation_name(
                chosen["relation"]
            )
        )

        truly_visited_answers = (
            visited & gold
        )

        if true_complete_before:
            over_search_actions += 1

    # Fixed-budget learned ranker result.
    fixed_result = create_result(
        qid=qid,
        policy="model_fixed_e2e",
        gold_answers=gold,
        predicted_answers=
            predicted_answers,
        truly_visited_answers=
            truly_visited_answers,
        actions=actions,
        retrieved_edges=
            retrieved_edges,
        over_search_actions=
            over_search_actions,
        stop_reason=(
            "action_budget"
            if actions >= MAX_ACTIONS
            else "empty_frontier"
        ),
        premature_stop=False,
        frontier_truncations=
            frontier_truncations,
    )

    # SetStop follows the identical trajectory until
    # the first frozen-threshold stopping decision.
    stopping_snapshot = next(
        (
            snapshot
            for snapshot in snapshots
            if snapshot["should_stop"]
        ),
        None
    )

    if stopping_snapshot is None:
        setstop_result = create_result(
            qid=qid,
            policy="setstop_e2e",
            gold_answers=gold,
            predicted_answers=
                predicted_answers,
            truly_visited_answers=
                truly_visited_answers,
            actions=actions,
            retrieved_edges=
                retrieved_edges,
            over_search_actions=
                over_search_actions,
            stop_reason=(
                "action_budget"
                if actions >= MAX_ACTIONS
                else "empty_frontier"
            ),
            premature_stop=False,
            frontier_truncations=
                frontier_truncations,
        )
    else:
        setstop_result = create_result(
            qid=qid,
            policy="setstop_e2e",
            gold_answers=gold,
            predicted_answers=
                stopping_snapshot[
                    "predicted_answers"
                ],
            truly_visited_answers=
                stopping_snapshot[
                    "truly_visited_answers"
                ],
            actions=stopping_snapshot[
                "actions"
            ],
            retrieved_edges=
                stopping_snapshot[
                    "retrieved_edges"
                ],
            over_search_actions=
                stopping_snapshot[
                    "over_search_actions"
                ],
            stop_reason="learned_stop",
            premature_stop=
                stopping_snapshot[
                    "premature_if_stopped"
                ],
            frontier_truncations=
                stopping_snapshot[
                    "frontier_truncations"
                ],
        )

    return fixed_result, setstop_result


# ============================================================
# 11. NON-CONTROLLER POLICIES
# ============================================================

def run_baseline_policy(
    qid,
    example,
    policy
):
    question = get_question(example)
    gold = get_gold_ids(example)
    topics = get_topic_ids(example)

    _, edges = get_graph(example)

    adjacency = build_grouped_adjacency(
        edges
    )

    visited = set(topics)

    node_depth = {
        topic: 0
        for topic in topics
    }

    node_paths = {
        topic: []
        for topic in topics
    }

    expanded = set()

    node_answer_scores = {}
    predicted_answers = set()

    truly_visited_answers = (
        visited & gold
    )

    actions = 0
    retrieved_edges = 0
    over_search_actions = 0
    zero_prediction_streak = 0
    frontier_truncations = 0

    stop_reason = None
    premature_stop = False

    rng = random.Random(
        stable_seed(f"{qid}_{policy}")
    )

    while actions < MAX_ACTIONS:
        all_branches = get_frontier(
            visited,
            node_depth,
            expanded,
            adjacency,
        )

        if not all_branches:
            stop_reason = "empty_frontier"
            break

        if (
            len(all_branches)
            > MAX_FRONTIER_CANDIDATES
        ):
            frontier_truncations += 1

        retrieved_branches = retrieve_frontier(
            all_branches,
            question
        )

        if policy in [
            "relation_fixed_e2e",
            "patience2_e2e",
        ]:
            chosen = retrieved_branches[0]

        elif policy == "fifo_fixed_e2e":
            chosen = min(
                all_branches,
                key=lambda branch: (
                    branch["source_depth"],
                    branch["relation"],
                    branch["source"],
                )
            )

        elif policy == "random_fixed_e2e":
            chosen = all_branches[
                rng.randrange(
                    len(all_branches)
                )
            ]

        elif policy == "oracle_rank_e2e":
            unseen_gold = (
                gold
                - truly_visited_answers
            )

            scored = []

            for branch in all_branches:
                reachable = (
                    reachable_gold_from_branch(
                        branch,
                        adjacency,
                        expanded,
                        unseen_gold,
                    )
                )

                scored.append(
                    (
                        len(reachable),
                        branch,
                    )
                )

            scored.sort(
                key=lambda item: (
                    -item[0],
                    item[1]["source_depth"],
                    -item[1]["target_count"],
                    item[1]["relation"],
                    item[1]["source"],
                )
            )

            oracle_gain, chosen = scored[0]

            if oracle_gain == 0:
                stop_reason = "oracle_stop"
                break

        else:
            raise ValueError(
                f"Unknown policy: {policy}"
            )

        true_complete_before = (
            truly_visited_answers == gold
        )

        predicted_count_before = len(
            predicted_answers
        )

        expand_branch(
            chosen,
            visited,
            node_depth,
            node_paths,
            expanded,
        )

        recognize_branch_answers(
            question=question,
            branch=chosen,
            node_paths=node_paths,
            adjacency=adjacency,
            node_answer_scores=
                node_answer_scores,
            predicted_answers=
                predicted_answers,
        )

        actions += 1
        retrieved_edges += (
            chosen["target_count"]
        )

        truly_visited_answers = (
            visited & gold
        )

        if true_complete_before:
            over_search_actions += 1

        if (
            len(predicted_answers)
            == predicted_count_before
        ):
            zero_prediction_streak += 1
        else:
            zero_prediction_streak = 0

        if (
            policy == "patience2_e2e"
            and zero_prediction_streak >= 2
        ):
            stop_reason = "patience_2"

            current_branches = get_frontier(
                visited,
                node_depth,
                expanded,
                adjacency,
            )

            premature_stop = (
                remaining_gold_is_reachable(
                    branches=current_branches,
                    adjacency=adjacency,
                    expanded=expanded,
                    gold=gold,
                    truly_visited_answers=
                        truly_visited_answers,
                )
            )

            break

    if stop_reason is None:
        stop_reason = (
            "action_budget"
            if actions >= MAX_ACTIONS
            else "empty_frontier"
        )

    return create_result(
        qid=qid,
        policy=policy,
        gold_answers=gold,
        predicted_answers=
            predicted_answers,
        truly_visited_answers=
            truly_visited_answers,
        actions=actions,
        retrieved_edges=retrieved_edges,
        over_search_actions=
            over_search_actions,
        stop_reason=stop_reason,
        premature_stop=premature_stop,
        frontier_truncations=
            frontier_truncations,
    )


# ============================================================
# 12. LOAD TEST QUESTIONS
# ============================================================

test_examples = []

for line_number, example in enumerate(
    iter_jsonl(
        DATA_DIR / "test_simple.json"
    ),
    start=1
):
    gold = get_gold_ids(example)
    topics = get_topic_ids(example)
    graph_nodes, graph_edges = (
        get_graph(example)
    )

    usable = (
        2 <= len(gold) <= MAX_ANSWERS
        and gold.issubset(graph_nodes)
        and bool(topics)
        and bool(graph_edges)
    )

    if usable:
        qid = get_question_id(
            "test",
            example,
            line_number
        )

        test_examples.append(
            (qid, example)
        )

assert len(test_examples) == 736, (
    f"Expected 736 questions, "
    f"found {len(test_examples)}"
)

print(
    "\nEnd-to-end test questions:",
    len(test_examples)
)


# ============================================================
# 13. RUN FINAL EVALUATION
# ============================================================

results = []

evaluation_start = time.time()

baseline_policies = [
    "fifo_fixed_e2e",
    "relation_fixed_e2e",
    "patience2_e2e",
    "random_fixed_e2e",
    "oracle_rank_e2e",
]

for question_number, (
    qid,
    example
) in enumerate(
    test_examples,
    start=1
):
    fixed_result, setstop_result = (
        run_learned_trajectory(
            qid,
            example
        )
    )

    results.extend([
        fixed_result,
        setstop_result,
    ])

    for policy in baseline_policies:
        results.append(
            run_baseline_policy(
                qid,
                example,
                policy,
            )
        )

    if question_number % 25 == 0:
        elapsed = (
            time.time()
            - evaluation_start
        ) / 60

        print(
            f"Questions: {question_number}/"
            f"{len(test_examples)} | "
            f"{elapsed:.1f} minutes"
        )


# ============================================================
# 14. AGGREGATION
# ============================================================

def bootstrap_mean_ci(
    values,
    samples=2000,
    seed=42
):
    values = np.asarray(
        values,
        dtype=np.float64
    )

    rng = np.random.default_rng(seed)

    means = np.empty(
        samples,
        dtype=np.float64
    )

    for index in range(samples):
        sampled_indices = rng.integers(
            0,
            len(values),
            len(values),
        )

        means[index] = np.mean(
            values[sampled_indices]
        )

    lower, upper = np.percentile(
        means,
        [2.5, 97.5]
    )

    return {
        "mean": float(np.mean(values)),
        "ci95_lower": float(lower),
        "ci95_upper": float(upper),
    }


grouped = collections.defaultdict(list)

for result in results:
    grouped[result["policy"]].append(
        result
    )


summary = {}

metric_names = [
    "precision",
    "recall",
    "answer_f1",
    "exact_match",
    "complete_recall",
    "incomplete_set",
    "actions",
    "retrieved_edges",
    "over_search_actions",
    "premature_stop",
    "frontier_truncations",
]

for policy, policy_results in grouped.items():
    total_tp = sum(
        row["true_positive_count"]
        for row in policy_results
    )

    total_fp = sum(
        row["false_positive_count"]
        for row in policy_results
    )

    total_fn = sum(
        row["false_negative_count"]
        for row in policy_results
    )

    micro_precision = (
        total_tp / (total_tp + total_fp)
        if total_tp + total_fp
        else 0.0
    )

    micro_recall = (
        total_tp / (total_tp + total_fn)
        if total_tp + total_fn
        else 0.0
    )

    micro_f1 = (
        2
        * micro_precision
        * micro_recall
        / (micro_precision + micro_recall)
        if micro_precision + micro_recall
        else 0.0
    )

    policy_summary = {
        "questions": len(policy_results),

        "micro_precision":
            float(micro_precision),

        "micro_recall":
            float(micro_recall),

        "micro_f1":
            float(micro_f1),

        "total_true_positives":
            int(total_tp),

        "total_false_positives":
            int(total_fp),

        "total_false_negatives":
            int(total_fn),

        "stop_reasons": dict(
            collections.Counter(
                row["stop_reason"]
                for row in policy_results
            )
        ),
    }

    for metric_name in metric_names:
        values = [
            row[metric_name]
            for row in policy_results
        ]

        policy_summary[metric_name] = (
            bootstrap_mean_ci(
                values,
                samples=2000,
                seed=stable_seed(
                    f"{policy}_{metric_name}"
                ),
            )
        )

    summary[policy] = policy_summary


# ============================================================
# 15. PAIRED COMPARISONS
# ============================================================

def paired_bootstrap(
    first_values,
    second_values,
    samples=5000,
    seed=42
):
    differences = (
        np.asarray(
            first_values,
            dtype=np.float64
        )
        -
        np.asarray(
            second_values,
            dtype=np.float64
        )
    )

    rng = np.random.default_rng(seed)

    means = np.empty(
        samples,
        dtype=np.float64
    )

    for index in range(samples):
        sampled_indices = rng.integers(
            0,
            len(differences),
            len(differences),
        )

        means[index] = np.mean(
            differences[sampled_indices]
        )

    lower, upper = np.percentile(
        means,
        [2.5, 97.5]
    )

    return {
        "mean_difference":
            float(np.mean(differences)),
        "ci95_lower": float(lower),
        "ci95_upper": float(upper),
        "significant_at_0_05": bool(
            lower > 0 or upper < 0
        ),
    }


by_policy_qid = {
    policy: {
        row["qid"]: row
        for row in policy_results
    }
    for policy, policy_results
    in grouped.items()
}

paired_comparisons = {}

for baseline in [
    "fifo_fixed_e2e",
    "relation_fixed_e2e",
    "patience2_e2e",
    "random_fixed_e2e",
    "model_fixed_e2e",
]:
    paired_comparisons[baseline] = {}

    for metric_name in [
        "precision",
        "recall",
        "answer_f1",
        "exact_match",
        "actions",
        "retrieved_edges",
    ]:
        setstop_values = []
        baseline_values = []

        for qid, setstop_row in (
            by_policy_qid[
                "setstop_e2e"
            ].items()
        ):
            baseline_row = (
                by_policy_qid[
                    baseline
                ][qid]
            )

            setstop_values.append(
                setstop_row[metric_name]
            )

            baseline_values.append(
                baseline_row[metric_name]
            )

        paired_comparisons[
            baseline
        ][metric_name] = (
            paired_bootstrap(
                setstop_values,
                baseline_values,
                samples=5000,
                seed=stable_seed(
                    f"{baseline}_"
                    f"{metric_name}"
                ),
            )
        )


# ============================================================
# 16. SAVE OUTPUTS
# ============================================================

per_question_path = (
    OUTPUT_ROOT
    / "end_to_end_per_question_v1.jsonl"
)

with open(
    per_question_path,
    "w",
    encoding="utf-8"
) as file:
    for row in results:
        file.write(
            json.dumps(
                row,
                ensure_ascii=False
            )
            + "\n"
        )


summary_path = (
    OUTPUT_ROOT
    / "end_to_end_summary_v1.json"
)

summary_path.write_text(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8",
)


paired_path = (
    OUTPUT_ROOT
    / "end_to_end_paired_comparisons_v1.json"
)

paired_path.write_text(
    json.dumps(
        paired_comparisons,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8",
)


previous_oracle_results = None

if ORACLE_EVALUATION_PATH.exists():
    previous_oracle_results = json.loads(
        ORACLE_EVALUATION_PATH.read_text(
            encoding="utf-8"
        )
    )


complete_output = {
    "evaluation_setting": (
        "fully end-to-end learned frontier "
        "ranking, stopping and answer recognition"
    ),
    "test_questions":
        len(test_examples),
    "controller_threshold":
        CONTROLLER_THRESHOLD,
    "answer_threshold":
        ANSWER_THRESHOLD,
    "maximum_actions":
        MAX_ACTIONS,
    "maximum_frontier_candidates":
        MAX_FRONTIER_CANDIDATES,
    "summary": summary,
    "paired_comparisons":
        paired_comparisons,
    "oracle_answer_recognition_reference":
        previous_oracle_results,
    "runtime_minutes": float(
        (time.time() - evaluation_start) / 60
    ),
}

complete_path = (
    OUTPUT_ROOT
    / "complete_end_to_end_evaluation_v1.json"
)

complete_path.write_text(
    json.dumps(
        complete_output,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8",
)


csv_path = (
    OUTPUT_ROOT
    / "end_to_end_results_table_v1.csv"
)

with open(
    csv_path,
    "w",
    newline="",
    encoding="utf-8"
) as file:
    writer = csv.writer(file)

    writer.writerow([
        "Policy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1",
        "Exact Match",
        "Complete Recall",
        "Micro F1",
        "Actions",
        "Retrieved Edges",
        "Premature Stop",
    ])

    for policy, policy_summary in summary.items():
        writer.writerow([
            policy,
            policy_summary[
                "precision"
            ]["mean"],
            policy_summary[
                "recall"
            ]["mean"],
            policy_summary[
                "answer_f1"
            ]["mean"],
            policy_summary[
                "exact_match"
            ]["mean"],
            policy_summary[
                "complete_recall"
            ]["mean"],
            policy_summary[
                "micro_f1"
            ],
            policy_summary[
                "actions"
            ]["mean"],
            policy_summary[
                "retrieved_edges"
            ]["mean"],
            policy_summary[
                "premature_stop"
            ]["mean"],
        ])


# ============================================================
# 17. FINAL TABLE
# ============================================================

policy_order = [
    "fifo_fixed_e2e",
    "relation_fixed_e2e",
    "patience2_e2e",
    "random_fixed_e2e",
    "model_fixed_e2e",
    "setstop_e2e",
    "oracle_rank_e2e",
]

print("\n" + "=" * 120)
print("FULLY END-TO-END TEST RESULTS")
print("=" * 120)

print(
    f"{'Policy':<22}"
    f"{'Precision':>11}"
    f"{'Recall':>10}"
    f"{'F1':>10}"
    f"{'Exact':>10}"
    f"{'Complete':>11}"
    f"{'Actions':>11}"
    f"{'Edges':>12}"
)

print("-" * 120)

for policy in policy_order:
    row = summary[policy]

    print(
        f"{policy:<22}"
        f"{row['precision']['mean']:>11.4f}"
        f"{row['recall']['mean']:>10.4f}"
        f"{row['answer_f1']['mean']:>10.4f}"
        f"{row['exact_match']['mean']:>10.4f}"
        f"{row['complete_recall']['mean']:>11.4f}"
        f"{row['actions']['mean']:>11.2f}"
        f"{row['retrieved_edges']['mean']:>12.2f}"
    )

print("=" * 120)

print("\nFrozen thresholds:")
print("Controller:", CONTROLLER_THRESHOLD)
print("Answer recognizer:", ANSWER_THRESHOLD)

print("\nSaved outputs:")
print(per_question_path)
print(summary_path)
print(paired_path)
print(complete_path)
print(csv_path)

print("\nDo not change either threshold based on test results.")
print("=" * 120)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 26.3 MB/s eta 0:00:00
Mounted at /content/drive
SETSTOP-KGQA FULL END-TO-END EVALUATION
Transformers: 5.17.0
PyTorch: 2.11.0+cu128
CUDA: True
Controller threshold: 0.7100000000000001
Answer threshold: 0.6699999999999999
Maximum actions: 12
Frontier candidates: 16
GPU: Tesla T4


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Models loaded successfully.

End-to-end test questions: 736
Questions: 25/736 | 1.1 minutes
Questions: 50/736 | 2.4 minutes
Questions: 75/736 | 3.6 minutes
Questions: 100/736 | 4.8 minutes
Questions: 125/736 | 6.0 minutes
Questions: 150/736 | 7.2 minutes
Questions: 175/736 | 8.3 minutes
Questions: 200/736 | 9.7 minutes
Questions: 225/736 | 10.8 minutes
Questions: 250/736 | 12.0 minutes
Questions: 275/736 | 13.2 minutes
Questions: 300/736 | 14.2 minutes
Questions: 325/736 | 15.3 minutes
Questions: 350/736 | 16.5 minutes
Questions: 375/736 | 17.4 minutes
Questions: 400/736 | 18.6 minutes
Questions: 425/736 | 19.8 minutes
Questions: 450/736 | 20.8 minutes
Questions: 475/736 | 22.0 minutes
Questions: 500/736 | 23.1 minutes
Questions: 525/736 | 24.3 minutes
Questions: 550/736 | 25.5 minutes
Questions: 575/736 | 26.6 minutes
Questions: 600/736 | 27.8 minutes
Questions: 625/736 | 28.9 minutes
Questions: 650/736 | 30.2 minutes
Questions: 675/736 | 31.4 minutes
Questions: 700/736 | 32.7 minute